# GEE_LULC_SVM walkthrough

Interactive demo of the local Python pipeline. Reproduces what the
command-line scripts do, but in cell-by-cell form so you can inspect
intermediate results.

**Prerequisite:** you've already run `python ../00_authenticate.py --project YOUR_PROJECT`
and uploaded your training samples as GEE assets.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))  # so we can `import lib`

import ee
import importlib.util as _ilu

# Edit this to your GCP project
PROJECT = 'your-gcp-project-id'

ee.Initialize(project=PROJECT)
print('GEE ready.')

## 1. Load AOI and training samples

In [ ]:
from lib.composites import landsat_sr_annual_composite
from lib.indices import add_all_indices, add_texture_features, BANDS_WITH_TEXTURE
from lib.svm_classify import train_and_classify_svm

# Inline-load the load_samples helpers (the file has a numeric prefix)
_spec = _ilu.spec_from_file_location('load_samples', Path.cwd().parent / '01_load_samples.py')
load_samples = _ilu.module_from_spec(_spec)
_spec.loader.exec_module(load_samples)

aoi = load_samples.load_aoi()  # default Dabaoshan asset
samples = load_samples.load_samples_from_assets(
    water_asset='users/yourname/water',
    built_up_asset='users/yourname/builtUp',
    unrestored_asset='users/yourname/unrestoredLand',
    restoring_asset='users/yourname/restoring',
    stable_veg_asset='users/yourname/stableVegetation',
)
print('Samples per class:', load_samples.summarize(samples))

## 2. Build a single-year composite and inspect band stats

In [ ]:
YEAR = 2020
CLOUD_THRESHOLD = 20

sr_dict = landsat_sr_annual_composite(YEAR, CLOUD_THRESHOLD, aoi)
l89 = sr_dict['Landsat89_SR']

# Quick band-stats roundtrip ,  uses getInfo() so don't do this in tight loops
stats = l89.select(['Blue', 'NIR', 'SWIR1']).reduceRegion(
    reducer=ee.Reducer.mean(),
    geometry=aoi.geometry(),
    scale=30,
    maxPixels=int(1e9),
    tileScale=4,
).getInfo()
print(stats)

## 3. Add indices + textures, classify

In [ ]:
img = add_texture_features(
    add_all_indices(
        l89.select(['Blue', 'Green', 'Red', 'NIR', 'SWIR1', 'SWIR2'])
    )
).set('isDummy', l89.get('isDummy'))

result = train_and_classify_svm(
    img, 'Landsat89_SR', BANDS_WITH_TEXTURE, samples, aoi,
    use_balancing=True, gamma=1.0, cost=100.0,
)
print(f'OA = {result.oa:.4f}, Kappa = {result.kappa:.4f}')

## 4. Download to disk

In [ ]:
from lib.io_utils import download_image_to_local

out_path = Path.cwd().parent.parent / 'outputs' / f'{YEAR}_Landsat89_SR_demo.tif'
download_image_to_local(
    result.image.toByte(), out_path, region=aoi.geometry(), scale=30,
)

## 5. Open the result locally with rasterio

In [ ]:
import rasterio
import numpy as np

with rasterio.open(out_path) as src:
    arr = src.read(1)
    print('CRS:', src.crs)
    print('Shape:', arr.shape)
    print('Class counts:', dict(zip(*np.unique(arr, return_counts=True))))